In [16]:
import torch
import os
from transformers import BitsAndBytesConfig
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

In [6]:
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
bnb_config_llama = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

In [12]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [3]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"

In [8]:
config = AutoConfig.from_pretrained(model_id)
print(f"Capas: {config.num_hidden_layers}")
print(f"Hidden Size: {config.hidden_size}")
print(f"Cabezales de atención: {config.num_attention_heads}")
print(f"Vocab Size: {config.vocab_size}")

Capas: 28
Hidden Size: 3072
Cabezales de atención: 24
Vocab Size: 128256


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_id,
    quantization_config= bnb_config_llama,
    device_map="auto"
)

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
#Esta línea prepara el modelo par el entrenamiento con Lora
model = prepare_model_for_kbit_training(model)

NameError: name 'model' is not defined

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()